# 04 Model 2: RNN + Classifier
Training of architecture 2: the same frozen word embeddings as in model 1, a 2-layer RNN (LSTM or GRU) and a 2-layer classifier (ReLU non-linearity). The RNN and the classifier are trained. Experiments are tracked with Weights & Biases.

Load the word numbers, the labels and the embedding matrix that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only used once at the end, in `05_evaluation.ipynb`.

In [1]:
import numpy as np
import torch

data = np.load("features/model2.npz")
# word numbers of O1, O2, h1, h2 for every example, and the labels (0 = h1, 1 = h2)
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
# GloVe vectors: row i = vector of the word with number i (row 0 = padding)
embedding_matrix = torch.tensor(data["embedding_matrix"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)
print(embedding_matrix.shape, embedding_matrix.dtype)

torch.Size([16801, 4, 34]) torch.int64
torch.Size([16801]) torch.int64
torch.Size([15465, 300]) torch.float32


## Classifier
The same 2-layer classifier as for model 1 (`03_model1_embeddings.ipynb`, from the course exercise "PyTorch" (Week 1)): linear layer, ReLU, linear layer. In model 2, its input are the final hidden states of the RNN instead of the averaged word vectors.

In [2]:
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

## Model 2: RNN + classifier
The same LSTM (or GRU) reads each of the four texts (O1, O2, h1, h2) word by word. The final hidden state of its last layer, h_n, summarizes the text (course slides "Advanced RNNs", slide 16). The four final hidden states are put side by side, like the four sentence vectors in model 1, and the classifier makes the decision. The embedding layer contains the frozen GloVe vectors (`freeze=True`), and number 0 is the padding (`padding_idx=0`). Each text is passed to the RNN with its real length, the number of word numbers that are not 0, using `pack_padded_sequence` (course slides "Project Discussions" (Week 4), slide 17), so the RNN stops after the last real word and h_n is the state after that word (see the padding section in `02_preprocessing.ipynb`). With `batch_first=True` the input has the shape [batch size, sequence length, embedding dim], as on slide 14 of the course slides "Preliminaries".

In [3]:
from torch.nn.utils.rnn import pack_padded_sequence

class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        super().__init__()
        # embedding layer: the frozen GloVe vectors, row 0 = padding
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # size of one word vector (300)
        embedding_dim = embedding_matrix.shape[1]
        # 2-layer LSTM or GRU; batch_first=True: input shape [batch size, sequence length, embedding dim]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        # classifier on the 4 final hidden states side by side (O1, O2, h1, h2)
        self.classifier = Classifier(4 * hidden_dim, 2, inner_dim=inner_dim)

    def forward(self, x):
        # x: word numbers, shape [batch size, 4, 34] (O1, O2, h1, h2)
        features = []
        for i in range(4):
            # one text (O1, O2, h1 or h2) of every example in the batch: [batch size, 34]
            text = x[:, i]
            # real length of each text = number of word numbers that are not 0 (padding)
            lengths = (text != 0).sum(dim=1)
            # look up the GloVe vectors: [batch size, 34, 300]
            embedded = self.embedding(text)
            # give the RNN the real lengths, so it stops after the last real word
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            # the RNN reads the text; only the final hidden state is needed
            _, hidden = self.rnn(packed)
            # the LSTM returns (h_n, c_n), the GRU only h_n
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            # h_n of the last layer: [batch size, hidden dim]
            features.append(hidden[-1])
        # the 4 final hidden states side by side: [batch size, 4 * hidden dim]
        features = torch.concat(features, dim=-1)
        return self.classifier(features)

# same start weights every time (as in model 1)
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
print(model)

RNNClassifier(
  (embedding): Embedding(15465, 300, padding_idx=0)
  (rnn): LSTM(300, 128, num_layers=2, batch_first=True)
  (classifier): Classifier(
    (linear1): Linear(in_features=512, out_features=100, bias=True)
    (relu): ReLU()
    (linear2): Linear(in_features=100, out_features=2, bias=True)
  )
)


Test the untrained model on the first 5 training examples: one pair of scores (h1, h2) per example.

In [4]:
# 5 examples, each with 4 texts of 34 word numbers
x = X_train[:5]
# forward pass: one pair of scores per example
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 4, 34])
torch.Size([5, 2])
tensor([[0.0967, 0.0726],
        [0.0921, 0.0704],
        [0.0973, 0.0738],
        [0.0959, 0.0718],
        [0.0912, 0.0662]], grad_fn=<AddmmBackward0>)


## Loss, optimizer and batches
The same loss, optimizer and batches as for model 1 (`03_model1_embeddings.ipynb`), so that the difference between the two models comes from the RNN: cross-entropy loss on the 2 scores (`nn.CrossEntropyLoss` applies the softmax itself), the Adam optimizer, training batches of 32 examples (shuffled in every epoch; `drop_last=True` skips the last batch, which would contain a single example), and equal batches for measuring loss and accuracy: training data in 53 batches of 317, validation data in 10 batches of 100.

In [5]:
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

# cross-entropy loss on the 2 scores (applies the softmax itself)
criterion = nn.CrossEntropyLoss()

# training batches: 32 examples, shuffled in every epoch, the last batch (1 example) is skipped
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True, drop_last=True)
# equal batches for measuring loss and accuracy (16801 = 53 x 317, 1000 = 10 x 100), not shuffled
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

# number of batches, and the shape of one training batch: 32 examples, 4 texts, 34 word numbers
x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

525 53 10
torch.Size([32, 4, 34]) torch.Size([32])


## Evaluation
The same `evaluate()` as for model 1: the average of the batch losses, and the share of correct predictions (`argmax`, the hypothesis with the higher score) over all examples. A new untrained model is tested first: it should be at about 0.69 loss and 50% accuracy.

In [6]:
# same function as in 03_model1_embeddings.ipynb
def evaluate(model, loader):
    # evaluation mode, and no gradients are needed
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            # loss of this batch
            losses.append(criterion(output, y).item())
            # predicted class: the hypothesis with the higher score
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    # join the batches: one prediction and one label per example
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    # share of correct predictions
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

# untrained model with the same start weights as the training runs
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6931, accuracy: 50.80%


## Check: overfit one batch
Before the long training runs, check that the model can learn at all: a fresh model is trained on one fixed batch of 32 training examples for 50 steps. With about 400,000 trainable parameters, the model can learn 32 examples by heart, so the loss on this batch must go down to almost 0 and the accuracy up to 100%. If it does not, there is a bug, for example no gradient flowing through the packed sequences ("overfit one batch", Karpathy, 2019). At the end, check that the embedding layer still contains exactly the GloVe vectors, i.e. that the embeddings are really frozen.

In [7]:
# fresh model with the same start weights as the training runs
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
optimizer = Adam(model.parameters(), lr=0.001)
# one fixed batch: the first 32 training examples
x, y = X_train[:32], y_train[:32]

model.train()
for step in range(1, 51):
    # one training step, as for model 1
    output = model(x)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # loss and accuracy on the batch every 10 steps
    if step % 10 == 0:
        accuracy = (output.argmax(dim=1) == y).float().mean().item()
        print(f"Step {step}: loss {loss.item():.4f}, accuracy {accuracy:.2%}")

# frozen embeddings: the embedding layer must still be exactly the GloVe matrix
print("Embeddings unchanged:", torch.equal(model.embedding.weight, embedding_matrix))

Step 10: loss 0.6012, accuracy 56.25%
Step 20: loss 0.0390, accuracy 100.00%
Step 30: loss 0.0001, accuracy 100.00%
Step 40: loss 0.0000, accuracy 100.00%
Step 50: loss 0.0000, accuracy 100.00%
Embeddings unchanged: True


## Experiment tracking
The training runs are tracked with W&B, as for model 1 (same project `nlp-project1-art`).

In [8]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\mmici\_netrc.
wandb: Currently logged in as: maria-micioni (maria-micioni-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

## Training
The same training function as for model 1 (`03_model1_embeddings.ipynb`): every call trains a fresh model with seed 42 and creates one W&B run. In every epoch, one training step for each batch, then the loss and accuracy on the training and validation data, logged to W&B. The model is saved to `checkpoints/` whenever its validation loss is the lowest so far, and the weights of the best epoch are loaded again at the end. The best values are written to the W&B run summary, so that the runs can be sorted by their best validation loss. The hyperparameters are the type of RNN (`lstm` or `gru`), its hidden size and the learning rate.

New for model 2 is early stopping (course slides "Project Discussions" (Week 6), slide 38): the training stops when the validation loss has not improved for 3 epochs in a row (patience 3), after at most 15 epochs. The validation loss is used, as for the checkpoints. One epoch of model 2 takes several minutes on the CPU (model 1: a few seconds), so early stopping saves a lot of time: the epochs after the best one are only needed to see that the validation loss does not improve any more. The time of each epoch is printed. No dropout is used (`dropout=0.0`), as in model 1, so that both models are trained the same way; the best checkpoint by validation loss keeps the model from before it overfits.

In [9]:
import os
import time

os.makedirs("checkpoints", exist_ok=True)

def train(rnn_type="lstm", hidden_dim=128, learning_rate=0.001, max_epochs=15, patience=3):
    # run name, e.g. glove-lstm-h128-lr0.001
    name = f"glove-{rnn_type}-h{hidden_dim}-lr{learning_rate}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art",
        name=name,
        config={
            "model": f"glove-{rnn_type}",
            "rnn_type": rnn_type,
            "hidden_dim": hidden_dim,
            "num_layers": 2,
            "inner_dim": 100,
            "dropout": 0.0,
            "learning_rate": learning_rate,
            "max_epochs": max_epochs,
            "patience": patience,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    # fresh model with the same start weights in every run
    torch.manual_seed(42)
    model = RNNClassifier(embedding_matrix, rnn_type=rnn_type, hidden_dim=hidden_dim)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        start = time.time()
        # one training step per batch, as for model 1
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        # loss and accuracy after this epoch
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        minutes = (time.time() - start) / 60
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%} ({minutes:.1f} min)")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                print(f"Early stopping: no lower validation loss in the last {patience} epochs")
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    # return the model of the best epoch, not the one of the last epoch
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

## Experiments
Hyperparameter tuning on the validation set, as for model 1. The experiments answer three questions:
1. **Does reading the words in order help?** Model 1 averages the word vectors of each text (best validation loss 0.6494, accuracy 62.70%). The RNN reads the words one after the other, so the word order counts.
2. **LSTM or GRU?** The GRU is a simpler version of the LSTM; it is a bit faster and easier to train and has a similar performance, and which one is better depends on the task and the dataset (course slides "Recurrent Neural Networks" (Week 3), slides 62 and 68).
3. **Is the learning rate 0.01 too high, as for model 1?** The learning rate is "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32). For model 1, 0.01 made most ReLU units inactive.

This gives 4 runs: LSTM and GRU, each with learning rate 0.001 and 0.01; hidden size 128, 2 layers, no dropout, batch size 32, seed 42. Each configuration is trained once (one seed), as for model 1, so differences of a few percentage points can be chance (standard error of the validation accuracy about 1.5 percentage points, see the conclusion of `03_model1_embeddings.ipynb`). The 4 runs take about 4 to 5 hours on the CPU, so this cell is run overnight.

In [ ]:
# 4 runs: LSTM and GRU, each with learning rate 0.001 and 0.01
for rnn_type in ["lstm", "gru"]:
    for learning_rate in [0.001, 0.01]:
        train(rnn_type=rnn_type, learning_rate=learning_rate)

Epoch 1: train loss 0.6658, train acc 66.06%, valid loss 0.6873, valid acc 62.60% (1.9 min)


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier` (same as in `03_model1_embeddings.ipynb`), based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Model 2: 2-layer RNN (LSTM or GRU, PyTorch implementations) with the frozen word embeddings of model 1 and a 2-layer classifier with ReLU, as required in the Project 1 description; `nn.Embedding.from_pretrained` with `freeze=True` and `padding_idx=0` from the PyTorch documentation `torch.nn.Embedding`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html; `nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); `nn.LSTM`, `nn.GRU` and `dropout` (between the RNN layers) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html and https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html; `batch_first=True` for the input shape [batch size, sequence length, embedding dim] from the course slides "Preliminaries" (Week 1), slide 14; final hidden states of the texts as input of the classifier from the course slides "Advanced RNNs" (Week 4), slides 16-17.
- Forward pass of model 2: `pack_padded_sequence` for padding in RNNs from the course slides "Project Discussions" (Week 4), slide 17, with the lengths on the CPU and `enforce_sorted=False` from the PyTorch documentation `torch.nn.utils.rnn.pack_padded_sequence`, https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html; final hidden state of the last layer `hn[-1]` and `torch.concat(..., dim=-1)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); output `(h_n, c_n)` of the LSTM and `h_n` of the GRU from the PyTorch documentation `torch.nn.LSTM` and `torch.nn.GRU`; `isinstance` from the Python documentation, https://docs.python.org/3/library/functions.html#isinstance; counting the non-zero numbers with `(text != 0).sum(dim=1)` (PyTorch documentation `torch.sum`, https://docs.pytorch.org/docs/stable/generated/torch.sum.html); test forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss, optimizer and batches: the same as in `03_model1_embeddings.ipynb`: cross-entropy loss from the course slides "Preliminaries" (Week 1), slide 27, and `nn.CrossEntropyLoss` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html; Adam from the same slides, slide 29; `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html.
- Evaluation: the same function as in `03_model1_embeddings.ipynb` (batch losses, `argmax` predictions and labels as in the validation of the course exercise "Experiment Tracking"; `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).
- Check with one batch: "overfit one batch" from Karpathy, A. (2019), *A Recipe for Training Neural Networks*, https://karpathy.github.io/2019/04/25/recipe/; training step as in the course exercise "PyTorch" (Week 1); `torch.equal` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.equal.html.
- Experiment tracking and training: the same as in `03_model1_embeddings.ipynb`: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; logging with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log; best values in the run summary as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary; training loop as in the PyTorch tutorial "Optimizing Model Parameters"; saving and loading only the parameters (`state_dict`) as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html, which is the recommended way on slide 36 of the course slides "Project Discussions" (Week 6). Early stopping on the validation loss with a patience from the course slides "Project Discussions" (Week 6), slide 38. Time per epoch with `time.time()` from the Python documentation, https://docs.python.org/3/library/time.html.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26"; LSTM vs. GRU from the course slides "Recurrent Neural Networks" (Week 3), slides 62 and 68; the learning rate as "the most important hyperparameter" from the course slides "Project Discussions" (Week 6), slide 32.
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.